In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Ambiente configurado com sucesso!")
print(f"Versão do pandas: {pd.__version__}")
print(f"Versão do NumPy: {np.__version__}")

Ambiente configurado com sucesso!
Versão do pandas: 3.0.6
Versão do NumPy: 2.4.6


In [2]:
# Caminho para a pasta que contém os dados brutos
DATA_DIR = Path("../data/raw")

# Caminhos das quatro bases
arquivos = {
    "queimadas": DATA_DIR / "queimadas_2024_2025.csv",
    "mapbiomas": DATA_DIR / "MapBiomas.csv",
    "estacoes": DATA_DIR / "BDMEP-estacao.csv",
    "meteorologia": DATA_DIR / "BDMEP-2025.csv"
}

# Verificar se todos os arquivos foram encontrados
for nome, caminho in arquivos.items():
    print(f"{nome:15} | Existe: {caminho.exists()} | {caminho.name}")

queimadas       | Existe: True | queimadas_2024_2025.csv
mapbiomas       | Existe: True | MapBiomas.csv
estacoes        | Existe: True | BDMEP-estacao.csv
meteorologia    | Existe: True | BDMEP-2025.csv


# Análise e Compreensão dos Dados

## Projeto Final — Previsão de Risco de Incêndios no Brasil

Neste notebook, será realizada a análise inicial das quatro bases de dados utilizadas no projeto.

O objetivo desta etapa é compreender a estrutura dos dados, identificar as principais variáveis, verificar tipos de dados, valores ausentes, duplicatas e possíveis relações entre as bases antes das etapas de tratamento e modelagem.

### Bases utilizadas

1. **Queimadas 2024–2025** — registros de focos de queimadas.
2. **MapBiomas** — informações territoriais e ambientais.
3. **BDMEP Estações** — informações das estações meteorológicas.
4. **BDMEP 2025** — medições meteorológicas.

## Inspeção inicial das bases

Nesta etapa, será realizada uma leitura de uma pequena amostra de cada base para identificar sua estrutura, colunas e formato antes do carregamento completo dos dados.

In [3]:
# Leitura inicial de apenas 5 linhas de cada base
# Isso evita carregar os arquivos grandes completamente na memória.

amostras = {}

for nome, caminho in arquivos.items():
    print(f"\n{'=' * 60}")
    print(f"BASE: {nome.upper()}")
    print(f"{'=' * 60}")

    try:
        df_sample = pd.read_csv(caminho, nrows=5)
        amostras[nome] = df_sample

        print(f"Quantidade de colunas: {df_sample.shape[1]}")
        print("\nColunas:")
        print(df_sample.columns.tolist())

        display(df_sample)

    except Exception as e:
        print(f"Erro ao ler {nome}: {e}")


BASE: QUEIMADAS
Quantidade de colunas: 1

Colunas:
['ano;mes;data_hora;bioma;sigla_uf;sigla_uf_nome;id_municipio;id_municipio_nome;latitude;longitude;satelite;dias_sem_chuva;precipitacao;risco_fogo;potencia_radiativa_fogo']


,ano;mes;data_hora;bioma;sigla_uf;sigla_uf_nome;id_municipio;id_municipio_nome;latitude;longitude;satelite;dias_sem_chuva;precipitacao;risco_fogo;potencia_radiativa_fogo
0,2024;9;2024-09-13 04:29:00;Mata Atlântica;MG;Minas Gerais;3166907;Serrania;-21.56454;-46.03047;NPP-375D;19.0;0.0;1.0;7.3
1,2024;9;2024-09-13 04:29:00;Mata Atlântica;ES;Espírito Santo;3202652;Irupi;-20.27151;-41.69187;NPP-375D;34.0;0.0;1.0;3.8
2,2024;9;2024-09-13 04:29:00;Mata Atlântica;RJ;Rio de Janeiro;3300308;Barra do Piraí;-22.42786;-43.81413;NPP-375D;18.0;0.0;1.0;2.5
3,2024;9;2024-09-13 04:29:00;Mata Atlântica;RJ;Rio de Janeiro;3303708;Paraíba do Sul;-22.18328;-43.35031;NPP-375D;18.0;0.0;1.0;26.5
4,2024;9;2024-09-13 04:29:00;Mata Atlântica;RJ;Rio de Janeiro;3303906;Petrópolis;-22.53255;-43.13242;NPP-375D;-999.0;0.0;1.0;2.0



BASE: MAPBIOMAS
Quantidade de colunas: 7

Colunas:
['ano', 'sigla_uf', 'sigla_uf_nome', 'id_municipio', 'id_municipio_nome', 'id_classe', 'area']


,ano,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,id_classe,area
0,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,3,5107.484730
1,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,4,270.661602
2,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,11,7.426764
3,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,12,937.381233
4,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,15,613.888604



BASE: ESTACOES
Quantidade de colunas: 8

Colunas:
['id_municipio', 'id_municipio_nome', 'id_estacao', 'estacao', 'data_fundacao', 'latitude', 'longitude', 'altitude']


,id_municipio,id_municipio_nome,id_estacao,estacao,data_fundacao,latitude,longitude,altitude
0,5300108,Brasília,A001,Brasilia,2000-05-07,-15.789444,-47.925833,1160.96
1,5300108,Brasília,A042,Brazlandia,2017-07-19,-15.599722,-48.131111,1143.00
2,5300108,Brasília,A045,Aguas Emendadas,2008-10-03,-15.596491,-47.625801,1030.36
3,5300108,Brasília,A046,Gama (Ponte Alta),2014-10-01,-15.935278,-48.137500,990.00
4,5300108,Brasília,A047,Paranoa (Coopa-Df),2017-02-07,-16.012222,-47.557417,1043.00



BASE: METEOROLOGIA
Quantidade de colunas: 21

Colunas:
['ano', 'data', 'hora', 'id_estacao', 'precipitacao_total', 'pressao_atm_hora', 'pressao_atm_max', 'pressao_atm_min', 'radiacao_global', 'temperatura_bulbo_hora', 'temperatura_orvalho_hora', 'temperatura_max', 'temperatura_min', 'temperatura_orvalho_max', 'temperatura_orvalho_min', 'umidade_rel_max', 'umidade_rel_min', 'umidade_rel_hora', 'vento_direcao', 'vento_rajada_max', 'vento_velocidade']


,ano,data,hora,id_estacao,precipitacao_total,pressao_atm_hora,pressao_atm_max,pressao_atm_min,radiacao_global,temperatura_bulbo_hora,temperatura_orvalho_hora,temperatura_max,temperatura_min,temperatura_orvalho_max,temperatura_orvalho_min,umidade_rel_max,umidade_rel_min,umidade_rel_hora,vento_direcao,vento_rajada_max,vento_velocidade
0,2025,2025-05-29,13:00:00,A316,0.0,997.6,997.9,997.5,2523.7,30.7,16.0,31.1,28.8,18.5,15.9,53.0,41.0,41.0,106.0,9.1,4.2
1,2025,2025-01-21,16:00:00,A867,0.0,1007.7,1008.4,1007.7,3896.8,32.6,17.6,32.8,31.6,18.6,14.8,46.0,34.0,41.0,256.0,8.8,4.5
2,2025,2025-05-14,18:00:00,A366,0.0,958.9,959.3,958.7,2227.9,30.3,11.8,31.2,29.5,13.7,11.8,36.0,31.0,32.0,119.0,11.7,5.5
3,2025,2025-05-27,16:00:00,A849,0.0,971.3,972.2,971.3,2279.0,29.8,15.3,29.8,27.9,15.4,14.5,45.0,41.0,41.0,NaN,10.3,5.5
4,2025,2025-04-01,19:00:00,A349,0.0,959.8,959.9,959.7,1831.7,33.6,16.2,34.3,33.3,16.3,15.4,35.0,33.0,35.0,165.0,10.7,5.2


In [8]:
# Inspeção das bases usando detecção automática do separador

amostras = {}

for nome, caminho in arquivos.items():
    print(f"\n{'=' * 60}")
    print(f"BASE: {nome.upper()}")
    print(f"{'=' * 60}")

    try:
        # sep=None tenta detectar automaticamente o separador
        df_sample = pd.read_csv(
            caminho,
            nrows=5,
            sep=None,
            engine="python"
        )

        amostras[nome] = df_sample

        print(f"Quantidade de colunas: {df_sample.shape[1]}")
        print("\nColunas:")
        print(df_sample.columns.tolist())

        display(df_sample)

    except Exception as e:
        print(f"Erro ao ler {nome}: {e}")


BASE: QUEIMADAS
Quantidade de colunas: 15

Colunas:
['\ufeffano', 'mes', 'data_hora', 'bioma', 'sigla_uf', 'sigla_uf_nome', 'id_municipio', 'id_municipio_nome', 'latitude', 'longitude', 'satelite', 'dias_sem_chuva', 'precipitacao', 'risco_fogo', 'potencia_radiativa_fogo']


,﻿ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,9,2024-09-13 04:29:00,Mata Atlântica,MG,Minas Gerais,3166907,Serrania,-21.56454,-46.03047,NPP-375D,19.0,0.0,1.0,7.3
1,2024,9,2024-09-13 04:29:00,Mata Atlântica,ES,Espírito Santo,3202652,Irupi,-20.27151,-41.69187,NPP-375D,34.0,0.0,1.0,3.8
2,2024,9,2024-09-13 04:29:00,Mata Atlântica,RJ,Rio de Janeiro,3300308,Barra do Piraí,-22.42786,-43.81413,NPP-375D,18.0,0.0,1.0,2.5
3,2024,9,2024-09-13 04:29:00,Mata Atlântica,RJ,Rio de Janeiro,3303708,Paraíba do Sul,-22.18328,-43.35031,NPP-375D,18.0,0.0,1.0,26.5
4,2024,9,2024-09-13 04:29:00,Mata Atlântica,RJ,Rio de Janeiro,3303906,Petrópolis,-22.53255,-43.13242,NPP-375D,-999.0,0.0,1.0,2.0



BASE: MAPBIOMAS
Quantidade de colunas: 7

Colunas:
['ano', 'sigla_uf', 'sigla_uf_nome', 'id_municipio', 'id_municipio_nome', 'id_classe', 'area']


,ano,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,id_classe,area
0,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,3,5107.484730
1,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,4,270.661602
2,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,11,7.426764
3,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,12,937.381233
4,1989,RO,Rondônia,1100015,Alta Floresta D'Oeste,15,613.888604



BASE: ESTACOES
Quantidade de colunas: 8

Colunas:
['id_municipio', 'id_municipio_nome', 'id_estacao', 'estacao', 'data_fundacao', 'latitude', 'longitude', 'altitude']


,id_municipio,id_municipio_nome,id_estacao,estacao,data_fundacao,latitude,longitude,altitude
0,5300108,Brasília,A001,Brasilia,2000-05-07,-15.789444,-47.925833,1160.96
1,5300108,Brasília,A042,Brazlandia,2017-07-19,-15.599722,-48.131111,1143.00
2,5300108,Brasília,A045,Aguas Emendadas,2008-10-03,-15.596491,-47.625801,1030.36
3,5300108,Brasília,A046,Gama (Ponte Alta),2014-10-01,-15.935278,-48.137500,990.00
4,5300108,Brasília,A047,Paranoa (Coopa-Df),2017-02-07,-16.012222,-47.557417,1043.00



BASE: METEOROLOGIA
Quantidade de colunas: 21

Colunas:
['ano', 'data', 'hora', 'id_estacao', 'precipitacao_total', 'pressao_atm_hora', 'pressao_atm_max', 'pressao_atm_min', 'radiacao_global', 'temperatura_bulbo_hora', 'temperatura_orvalho_hora', 'temperatura_max', 'temperatura_min', 'temperatura_orvalho_max', 'temperatura_orvalho_min', 'umidade_rel_max', 'umidade_rel_min', 'umidade_rel_hora', 'vento_direcao', 'vento_rajada_max', 'vento_velocidade']


,ano,data,hora,id_estacao,precipitacao_total,pressao_atm_hora,pressao_atm_max,pressao_atm_min,radiacao_global,temperatura_bulbo_hora,temperatura_orvalho_hora,temperatura_max,temperatura_min,temperatura_orvalho_max,temperatura_orvalho_min,umidade_rel_max,umidade_rel_min,umidade_rel_hora,vento_direcao,vento_rajada_max,vento_velocidade
0,2025,2025-05-29,13:00:00,A316,0.0,997.6,997.9,997.5,2523.7,30.7,16.0,31.1,28.8,18.5,15.9,53.0,41.0,41.0,106.0,9.1,4.2
1,2025,2025-01-21,16:00:00,A867,0.0,1007.7,1008.4,1007.7,3896.8,32.6,17.6,32.8,31.6,18.6,14.8,46.0,34.0,41.0,256.0,8.8,4.5
2,2025,2025-05-14,18:00:00,A366,0.0,958.9,959.3,958.7,2227.9,30.3,11.8,31.2,29.5,13.7,11.8,36.0,31.0,32.0,119.0,11.7,5.5
3,2025,2025-05-27,16:00:00,A849,0.0,971.3,972.2,971.3,2279.0,29.8,15.3,29.8,27.9,15.4,14.5,45.0,41.0,41.0,NaN,10.3,5.5
4,2025,2025-04-01,19:00:00,A349,0.0,959.8,959.9,959.7,1831.7,33.6,16.2,34.3,33.3,16.3,15.4,35.0,33.0,35.0,165.0,10.7,5.2


In [10]:
# Verificar tamanho das bases sem carregar tudo na memória

for nome, caminho in arquivos.items():
    print(f"\n{'=' * 50}")
    print(f"BASE: {nome.upper()}")

    try:
        # Detectar estrutura usando apenas uma pequena amostra
        amostra = pd.read_csv(
            caminho,
            sep=None,
            engine="python",
            nrows=100
        )

        # Contar linhas diretamente no arquivo
        with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
            total_linhas = sum(1 for _ in f) - 1

        print(f"Linhas: {total_linhas:,}")
        print(f"Colunas: {amostra.shape[1]}")

    except Exception as e:
        print(f"Erro: {e}")


BASE: QUEIMADAS
Linhas: 8,226,944
Colunas: 15

BASE: MAPBIOMAS
Linhas: 2,286,656
Colunas: 7

BASE: ESTACOES
Linhas: 612
Colunas: 8

BASE: METEOROLOGIA
Linhas: 4,891,176
Colunas: 21


In [11]:
# Análise inicial em chunks das bases grandes

for nome, caminho in arquivos.items():
    print(f"\n{'=' * 60}")
    print(f"BASE: {nome.upper()}")
    print("=" * 60)

    try:
        # Amostra de até 100.000 registros
        df = pd.read_csv(
            caminho,
            sep=None,
            engine="python",
            nrows=100_000
        )

        print(f"\nDimensão da amostra: {df.shape}")

        print("\nTipos de dados:")
        print(df.dtypes)

        print("\nValores ausentes:")
        ausentes = df.isnull().sum()
        print(ausentes[ausentes > 0].sort_values(ascending=False))

        print(f"\nDuplicatas na amostra: {df.duplicated().sum():,}")

    except Exception as e:
        print(f"Erro ao analisar {nome}: {e}")


BASE: QUEIMADAS

Dimensão da amostra: (100000, 15)

Tipos de dados:
﻿ano                         int64
mes                          int64
data_hora                      str
bioma                          str
sigla_uf                       str
sigla_uf_nome                  str
id_municipio                 int64
id_municipio_nome              str
latitude                   float64
longitude                  float64
satelite                       str
dias_sem_chuva             float64
precipitacao               float64
risco_fogo                 float64
potencia_radiativa_fogo    float64
dtype: object

Valores ausentes:
potencia_radiativa_fogo    3804
dias_sem_chuva             2670
precipitacao               2670
risco_fogo                 2670
dtype: int64

Duplicatas na amostra: 0

BASE: MAPBIOMAS

Dimensão da amostra: (100000, 7)

Tipos de dados:
ano                    int64
sigla_uf                 str
sigla_uf_nome            str
id_municipio           int64
id_municipio_nome     

In [12]:
# Valores ausentes - QUEIMADAS

df = pd.read_csv(
    arquivos["queimadas"],
    sep=None,
    engine="python",
    nrows=100_000
)

print("VALORES AUSENTES - QUEIMADAS")
print("=" * 40)

ausentes = df.isna().sum()

print(ausentes)

VALORES AUSENTES - QUEIMADAS
﻿ano                          0
mes                           0
data_hora                     0
bioma                         0
sigla_uf                      0
sigla_uf_nome                 0
id_municipio                  0
id_municipio_nome             0
latitude                      0
longitude                     0
satelite                      0
dias_sem_chuva             2670
precipitacao               2670
risco_fogo                 2670
potencia_radiativa_fogo    3804
dtype: int64


In [13]:
# Valores ausentes - MAPBIOMAS

df_mapbiomas = pd.read_csv(
    arquivos["mapbiomas"],
    sep=None,
    engine="python",
    nrows=100_000
)

print("VALORES AUSENTES - MAPBIOMAS")
print("=" * 40)

print(df_mapbiomas.isna().sum())

print("\nDuplicatas:", df_mapbiomas.duplicated().sum())

VALORES AUSENTES - MAPBIOMAS
ano                  0
sigla_uf             0
sigla_uf_nome        0
id_municipio         0
id_municipio_nome    0
id_classe            0
area                 0
dtype: int64

Duplicatas: 0


In [14]:
# Análise de valores ausentes e duplicatas - ESTAÇÕES

df_estacoes = pd.read_csv(
    arquivos["estacoes"],
    sep=None,
    engine="python"
)

print("VALORES AUSENTES - ESTAÇÕES")
print("=" * 40)

print(df_estacoes.isna().sum())

print("\nDuplicatas:", df_estacoes.duplicated().sum())
print("\nDimensão:", df_estacoes.shape)

VALORES AUSENTES - ESTAÇÕES
id_municipio          2
id_municipio_nome     2
id_estacao            0
estacao               0
data_fundacao         0
latitude              0
longitude             0
altitude             17
dtype: int64

Duplicatas: 0

Dimensão: (612, 8)


In [15]:
# Valores ausentes e duplicatas - METEOROLOGIA

df_meteorologia = pd.read_csv(
    arquivos["meteorologia"],
    sep=None,
    engine="python",
    nrows=100_000
)

print("VALORES AUSENTES - METEOROLOGIA")
print("=" * 40)

print(df_meteorologia.isna().sum())

print("\nDuplicatas:", df_meteorologia.duplicated().sum())
print("\nDimensão da amostra:", df_meteorologia.shape)

VALORES AUSENTES - METEOROLOGIA
ano                             0
data                            0
hora                            0
id_estacao                      0
precipitacao_total          53064
pressao_atm_hora            43333
pressao_atm_max             43419
pressao_atm_min             43426
radiacao_global             69572
temperatura_bulbo_hora      43929
temperatura_orvalho_hora    49196
temperatura_max             44081
temperatura_min             44082
temperatura_orvalho_max     49385
temperatura_orvalho_min     49404
umidade_rel_max             49057
umidade_rel_min             49027
umidade_rel_hora            48890
vento_direcao               52973
vento_rajada_max            54166
vento_velocidade            54073
dtype: int64

Duplicatas: 1573

Dimensão da amostra: (100000, 21)


## Conclusões da inspeção inicial

A inspeção inicial mostrou que as quatro bases possuem características bastante diferentes e, por isso, precisarão de estratégias específicas de preparação.

### Base de queimadas
A base de queimadas é a principal fonte de informação sobre as ocorrências de fogo. Ela possui mais de 8 milhões de registros e contém informações temporais, territoriais e indicadores relacionados às condições de fogo.

Na amostra analisada, foram identificados valores ausentes principalmente nas variáveis `dias_sem_chuva`, `precipitacao`, `risco_fogo` e `potencia_radiativa_fogo`. Também foram encontrados registros duplicados, que deverão ser investigados durante a etapa de tratamento.

### MapBiomas
A base do MapBiomas apresenta informações territoriais e de classes de uso/cobertura do solo por município. Não foram encontrados valores ausentes nem registros duplicados na análise realizada.

### Estações meteorológicas
A base de estações contém 612 registros. Foram identificados poucos valores ausentes: 2 em `id_municipio` e `id_municipio_nome`, além de 17 em `altitude`. Não foram encontrados registros duplicados.

### Dados meteorológicos
A base meteorológica possui aproximadamente 4,9 milhões de registros e apresenta uma quantidade significativa de valores ausentes em diversas variáveis meteorológicas, como precipitação, pressão atmosférica, radiação, temperatura, umidade e vento.

Por causa da elevada proporção de valores ausentes em algumas dessas variáveis, não será adequado simplesmente remover todas as linhas incompletas. Será necessário avaliar quais variáveis são realmente importantes para a previsão do risco de incêndio e definir uma estratégia de tratamento adequada.

### Próximos passos

A próxima etapa será dedicada à preparação dos dados, incluindo:

- análise das datas e períodos disponíveis em cada base;
- identificação e tratamento de duplicatas;
- análise dos valores ausentes;
- seleção das variáveis relevantes;
- padronização dos identificadores territoriais e temporais;
- definição da estratégia de integração entre as quatro bases.

Essa etapa será especialmente importante porque as bases possuem granularidades diferentes e não devem ser combinadas diretamente sem antes definir corretamente as chaves de integração.